# Exercise 3.2 — Auto-Generate a Weekly Report
### *The Friday-afternoon task that writes itself*

This is where a lot of the earlier skills come together. You have a week's worth of raw sales data sitting in a CSV. Instead of manually eyeballing it, filtering it in a spreadsheet, and typing up a summary every single week, you're going to write it once and run it every Friday in ten seconds.

**Why learn this instead of asking an AI assistant to summarise the CSV for you?**

You still can, for the *writing* part! But notice what this exercise actually builds: a completely reliable, repeatable calculation engine. Totals, top performer, best-selling product — these need to be *exactly right* every time, not "probably right." Code guarantees that in a way that re-prompting an AI each week doesn't. Once you have reliable numbers, *then* handing them to an AI to turn into prose is a great next step (that's exactly what you practised in Exercise 2.3).

## What you'll practise
- **Reading files** — pulling in a week of raw data
- **Processing data** — grouping, totalling, comparing
- **Decisions** — figuring out who/what "wins" this week
- **Producing something useful** — a clean, readable text report

## The scenario
`weekly_sales.csv` contains a week of sales transactions across 5 sales reps and 5 products. Let's turn it into a report ready to paste into an email or Slack update.

## Step 1: Load the data

In [1]:
import csv

with open("weekly_sales.csv", newline="") as f:
    reader = csv.DictReader(f)
    sales = list(reader)

for row in sales:
    row["amount"] = float(row["amount"])   # convert from text to number

print(f"Loaded {len(sales)} sales records")
print(sales[0])


Loaded 33 sales records
{'date': '2026-06-29', 'rep': 'Priya Nair', 'product': 'Pro Plan', 'amount': 206.89}


## Step 2: The headline number — total revenue this week

In [2]:
total_revenue = sum(row["amount"] for row in sales)
print(f"Total revenue this week: ${total_revenue:,.2f}")


Total revenue this week: $11,819.82


**New pattern: `sum(row["amount"] for row in sales)`.** This is a **generator expression** — a compact way of writing "go through every row, pull out the amount, and add them all up," without writing a full loop with a running total variable. It's the same idea as the loops you've written by hand in earlier exercises, just written more concisely once the pattern feels familiar. If it's not clear yet, that's completely fine — you can always write it as a full loop instead:

```python
total_revenue = 0
for row in sales:
    total_revenue += row["amount"]
```

Both versions do exactly the same thing.

## Step 3: Revenue by sales rep

In [3]:
revenue_by_rep = {}

for row in sales:
    rep = row["rep"]
    revenue_by_rep[rep] = revenue_by_rep.get(rep, 0) + row["amount"]

print(f"{'REP':<15}{'REVENUE'}")
print("-" * 25)
for rep, revenue in sorted(revenue_by_rep.items(), key=lambda pair: pair[1], reverse=True):
    print(f"{rep:<15}${revenue:,.2f}")


REP            REVENUE
-------------------------
Alex Tan       $3,344.35
Priya Nair     $3,052.92
Marcus Wong    $1,987.10
Farah Rahman   $1,768.03
Daniel Koh     $1,667.42


## Step 4: Who's the top performer this week?

In [4]:
# max() with a key function finds the item with the highest value according to that key
top_rep = max(revenue_by_rep, key=revenue_by_rep.get)
top_rep_revenue = revenue_by_rep[top_rep]

print(f"Top performer this week: {top_rep} (${top_rep_revenue:,.2f})")


Top performer this week: Alex Tan ($3,344.35)


**`max(revenue_by_rep, key=revenue_by_rep.get)`** looks a little dense, so let's unpack it: `revenue_by_rep` is a dictionary, and looping over a dictionary by default gives you its *keys* (the rep names). `key=revenue_by_rep.get` tells `max()` "when comparing these names, use their revenue (looked up via `.get`) to decide which one wins," rather than comparing the names alphabetically. It's the same "which value should I compare by?" idea you saw with `sorted(..., key=...)` in earlier exercises.

## Step 5: Best-selling product

In [5]:
revenue_by_product = {}
units_by_product = {}

for row in sales:
    product = row["product"]
    revenue_by_product[product] = revenue_by_product.get(product, 0) + row["amount"]
    units_by_product[product] = units_by_product.get(product, 0) + 1

top_product = max(revenue_by_product, key=revenue_by_product.get)

print(f"{'PRODUCT':<20}{'UNITS':<8}{'REVENUE'}")
print("-" * 40)
for product in revenue_by_product:
    print(f"{product:<20}{units_by_product[product]:<8}${revenue_by_product[product]:,.2f}")

print(f"\nBest seller: {top_product}")


PRODUCT             UNITS   REVENUE
----------------------------------------
Pro Plan            6       $2,879.07
Add-on: Support     6       $771.99
Enterprise Plan     8       $2,812.79
Starter Plan        5       $1,709.69
Add-on: Storage     8       $3,646.28

Best seller: Add-on: Storage


## Step 6: Put it all together into a proper report

In [6]:
from datetime import datetime as dt

dates = sorted(set(row["date"] for row in sales))
week_start, week_end = dates[0], dates[-1]

report_lines = []
report_lines.append(f"WEEKLY SALES REPORT ({week_start} to {week_end})")
report_lines.append("=" * 50)
report_lines.append(f"Total revenue: ${total_revenue:,.2f}")
report_lines.append(f"Total transactions: {len(sales)}")
report_lines.append("")
report_lines.append(f"Top performer: {top_rep} (${top_rep_revenue:,.2f})")
report_lines.append(f"Best-selling product: {top_product} ({units_by_product[top_product]} units, ${revenue_by_product[top_product]:,.2f})")
report_lines.append("")
report_lines.append("Revenue by rep:")
for rep, revenue in sorted(revenue_by_rep.items(), key=lambda pair: pair[1], reverse=True):
    report_lines.append(f"  - {rep}: ${revenue:,.2f}")
report_lines.append("")
report_lines.append("Revenue by product:")
for product, revenue in sorted(revenue_by_product.items(), key=lambda pair: pair[1], reverse=True):
    report_lines.append(f"  - {product}: ${revenue:,.2f}")

report = "\n".join(report_lines)
print(report)


WEEKLY SALES REPORT (2026-06-29 to 2026-07-05)
Total revenue: $11,819.82
Total transactions: 33

Top performer: Alex Tan ($3,344.35)
Best-selling product: Add-on: Storage (8 units, $3,646.28)

Revenue by rep:
  - Alex Tan: $3,344.35
  - Priya Nair: $3,052.92
  - Marcus Wong: $1,987.10
  - Farah Rahman: $1,768.03
  - Daniel Koh: $1,667.42

Revenue by product:
  - Add-on: Storage: $3,646.28
  - Pro Plan: $2,879.07
  - Enterprise Plan: $2,812.79
  - Starter Plan: $1,709.69
  - Add-on: Support: $771.99


In [7]:
with open("weekly_report_output.txt", "w") as f:
    f.write(report)

print("Saved weekly_report_output.txt - ready to paste into an email or Slack.")


Saved weekly_report_output.txt - ready to paste into an email or Slack.


## 🎉 You did it

You just built a Friday-afternoon report generator that reads raw data, calculates totals, finds top performers, and writes a clean summary — all in about 30 lines of code, all guaranteed to be arithmetically correct, every single week.

---

## 🧪 Try it yourself

1. Add a "week-over-week" comparison — hardcode last week's total as a variable and calculate the percentage change.
2. Break revenue down by **day of week** instead of by rep — which day brings in the most money?
3. Feed the `report` string from Step 6 into the prompt-building pattern from Exercise 2.3, and ask an AI assistant to turn it into a friendly two-paragraph summary for a non-technical stakeholder.

## 💡 Where this goes next
This exact structure — load data, group it, rank it, template it into a report — is the backbone of most internal "automated reporting" tools in real companies, whether they're written in Python, or wired up inside a no-code/low-code tool that's doing the same thing under the hood.